# Red Neuronal Titanic

Clasificacion con red neuronal multicapa en Titanic.

Conversion conceptual 1:1 desde el ejemplo R homologo.


In [ ]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "R").exists():
            return candidate
    raise FileNotFoundError("No se encontro la carpeta R del repositorio")

REPO_ROOT = find_repo_root(Path.cwd())
print("Repo root:", REPO_ROOT)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

data_path = REPO_ROOT / "R" / "nuevos" / "5_aprendizaje_supervisado" / "data" / "titanic.csv"
df = pd.read_csv(data_path)
y = df["Survived"]
X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ["Age", "Fare", "SibSp", "Parch", "Pclass"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["Sex", "Embarked"]),
])

model = Pipeline([
    ("pre", pre),
    ("clf", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1200, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))
